# Model inference with PyTorch Lightning

Loads the Lightning checkpoint and **StandardScaler** from `02_training_lightning.ipynb`, and the **LabelEncoder** from data preparation. Runs on `data/inference/input/input.csv`, verifies against `data/inference/expected/expected.csv`, and writes `data/inference/lightning/output_lightning.csv`.


## Import libraries


In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn

import lightning.pytorch as pl


## Config


In [ ]:
from config import (
    FEATURE_COLS,
    LIGHTNING_INFERENCE_CHECKPOINT_PATH,
    LIGHTNING_INFERENCE_DIR,
    LIGHTNING_INFERENCE_EXPECTED_PATH,
    LIGHTNING_INFERENCE_INPUT_PATH,
    LIGHTNING_INFERENCE_LABEL_ENCODING_PATH,
    LIGHTNING_INFERENCE_OUTPUT_PATH,
    LIGHTNING_INFERENCE_SCALER_JOBLIB_PATH,
)


## Lightning module (must match training)


In [ ]:
class IrisClassifier(pl.LightningModule):
    def __init__(self, num_features: int, num_classes: int, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.net = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


## Load checkpoint, scaler, and label encoding


In [ ]:
lit_model = IrisClassifier.load_from_checkpoint(
    LIGHTNING_INFERENCE_CHECKPOINT_PATH,
    map_location=torch.device("cpu"),
)
lit_model.eval()
print(f"Loaded checkpoint: {LIGHTNING_INFERENCE_CHECKPOINT_PATH}")

scaler = joblib.load(LIGHTNING_INFERENCE_SCALER_JOBLIB_PATH)
label_encoder = joblib.load(LIGHTNING_INFERENCE_LABEL_ENCODING_PATH)
print(f"Scaler: {LIGHTNING_INFERENCE_SCALER_JOBLIB_PATH}")
print(f"Label encoding: {LIGHTNING_INFERENCE_LABEL_ENCODING_PATH}")
print("Classes:", label_encoder.classes_)


## Load inference input


In [ ]:
df_input = pd.read_csv(LIGHTNING_INFERENCE_INPUT_PATH)
df_input[FEATURE_COLS] = df_input[FEATURE_COLS].astype(np.float32)
print(f"Input rows: {len(df_input)}")
df_input.head()


## Run inference


In [ ]:
X = df_input[FEATURE_COLS].values
X_scaled = scaler.transform(X)
x_tensor = torch.tensor(X_scaled, dtype=torch.float32)

with torch.no_grad():
    logits = lit_model(x_tensor)
    y_pred = torch.argmax(logits, dim=1).numpy()

species_pred = label_encoder.inverse_transform(y_pred)
df_out = df_input.copy()
df_out["species"] = np.asarray(species_pred, dtype=str)
df_out.head()


## Verify against expected.csv


In [ ]:
df_expected = pd.read_csv(LIGHTNING_INFERENCE_EXPECTED_PATH)
expected = df_expected["species"].astype(str).to_numpy()
predicted = df_out["species"].astype(str).to_numpy()

if np.array_equal(predicted, expected):
    print("Verification: predictions match expected.csv")
else:
    print("Verification: predictions do not match expected.csv")
    print("Expected:", expected)
    print("Got:     ", predicted)


## Save predictions


In [ ]:
os.makedirs(LIGHTNING_INFERENCE_DIR, exist_ok=True)
df_out.to_csv(LIGHTNING_INFERENCE_OUTPUT_PATH, index=False)
print(f"Saved: {LIGHTNING_INFERENCE_OUTPUT_PATH}")
